# 🔥 Mossad-pentast-ai - Cloud Model Fusion & Standalone Weight Merger

Fuses all trained LoRA security adapters (27,000 + 32 sources) directly into **Qwen2.5-7B-Instruct**.
- **Target Model**: `Mossad-pentast-ai`
- **Source Weights**: Node `mmossad2324` (`bugbounty_copilot_lora`)
- **Outputs**:
  1. Standalone 16-bit Float16 Model (`/kaggle/working/Mossad-pentast-ai`)
  2. Standalone GGUF Q4_K_M Quantized Model (`/kaggle/working/Mossad-pentast-ai-GGUF`)


In [ ]:
# 1. Install Unsloth and Core Dependencies
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Locate and Prepare LoRA Weights
import os, glob, json, urllib.request
LATEST_LORA_URLS = {"bugbounty_copilot_lora/README.md": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..7M_GJ4bKZUbMgxq8ejuyoQ.JJVetrH2OniYM_GCWipKGdxKgTlqYlArtn5m1nilSrq1mdfBSbdIOgRZ1e8FeNM0lCmNrKPe_mYh6K7OyNiK66CjHFwtxAD_qSfQOGrTWfcgPeM9pr2nof1vBz_1VcUIitr_JaofErU-EfCnta0m2eoaZicQwEp3imvFQIApUdM1o2c45bDGXL81ywxEtZ7wWBTKy9i-KxnVyBU-P4n6hsjjdU5i0xOWzhKuYfnuZA4JIhUuCHT4ZEq7ubZQb8HySqzee9X6jwdAHZxDOjT3z5LTEXF3RYPzZ4e4GgZQKFQbWrusCW-WZRxVLjcUDwzqGwHJjgaP23KvunSh7-Wb2k39QDeT8KLQjVZM34STMCezetfz37II3NGutmcH7KG4ASXDwMTrmtvs5wgaW8ZgXgTxkUzvwJwH16Vgl1a1BShiY7nNUzZ4QgZI3VgUKMLqPgws9TC204nFkdIKHAkMFi24-4wxF9DVM3EfS8RuCKYJvq57y4DBAExINdnjLDkqbnSrInBzQ4mClixUI1c4pzzBrDPO4BKoek7vCpO05YvzlVeKrs2CzlRGTuCDLkjgC4HCAt9x8alJYx7_Hv6VdWzZUIeilWIL-Fn3Dioq3SYqFXpopYWxGkyBcU9c5HAh.aqbDcpSZtJ6_JuI56rTqMw/bugbounty_copilot_lora/README.md", "bugbounty_copilot_lora/adapter_config.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..rnqnY7VgiOfohE0DJLb1Qw.TIcKO0jAJWYDxdresRksbZGoyIStq687oFxmjHf6a5khFwUVuZUKO0a5p9D_0FQeIj4b4zJXOVyWaVNiSXyCKiBD3BXo1iRXqmAVyRY0xIs7eamrygoo5shWaxXrA-MnuMXJDnNXAfXYNGurB27ORLNtU_CP9si0HKpkZ6lnnA2Zzdg_uRF5NVuVxadqdUcC9-XQR-c-MwXVWzoDzRJwtL7Bh8rTeRajfnG2itdi0wq7iLeRasNlgHvIf67tLVhymU8DSjlyJp9SnfFL5O0HivOyjT0FHRwX6wnl8CmQwBKIxkQWxZNz_cHP_eLmt--Wv77bGHoufdDQemuTrbG7MqNZPG_fERdiZQ8fmy5mzcw4Bht8aoKwdUCufyFeqgSF2bOOwhyCz76WuCGosfdcnnNaCs83tyN-mCoRAs0jJU1CArAgwbfMpAEHpDkI54tHbz9FDG7h18usjP5QPgDbGGjT3f82nfWmekGYp17WLJE3n6otfZyCa4cE_9bX2wco36eWWBeziSZve70dOqKSwlUGkZc_sX21uPl-wk_Kw688pjV5JSmwh1LegnY4eg2KM9-lwZ4fzIxZMB2v5Rb-aT-o9l85K2maAwfUe6Ob4flFx7L4nqaL8Z7RRtMGSdxY.YUZ3J5q9CQ_JVheTbh4zMQ/bugbounty_copilot_lora/adapter_config.json", "bugbounty_copilot_lora/adapter_model.safetensors": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..orlqneGbD4uwwWKeJRKEQw.x93U_uJYup2J1epkHgz2rHM7mlMz2ebz66Q9BWrTTuKp_vj1SbrS8kaVPzkm8cNM1S5cz90WHn70Ht22fqux9s_qAEwje__3dcPfyh6fidyf7uFvW81WOl2jM7H1MG_hqTU-oqAEafUmRlhiLu-BIuMW8jxN3wbqdgURy2CSmYrr1x1A0Na6KPadxf39vEvUa8z9SaQKOJZz8cAPAsmiZKsU_gtN204m9bMvoW5c5wI46lVKHpdjoK9LvZ9GsyUYj4vCIdx0Qg9OLWaGcUdIDqgf0dmHiwGO0eVJJO8-Jsh7YYr0RINgpcWH75p_jJgTC78qxB2ZWavzzV-xY1-z0xBagGdtJC9o9g_9D_dV7Rc73_bZLqHly50apUceFbvljXE6dmNafYufgJl61XSFtVoEiYcWDmgowvnJIgImWFLlo3tv4AD_Ww2bpC2HjfCWhddrOFXXPAzcq5Bspb2oTdi4v0_-sHji6Z7b-UJB_aS5PFGA3a_3mUs6fktOsL6evlXnAMGtVYf3FfV0nJXRILmy-aPQJFKK3MMpcI_kZ9SFMmqJca8UY03qt2byns5eI9HEO0CQG1SjNJjZnXF3XMvy8UeP-x5fjUnTfcXpURM3xpvff5wHm16nV6vZWptq._0aMH6WwCROluNnHeFBkvg/bugbounty_copilot_lora/adapter_model.safetensors", "bugbounty_copilot_lora/chat_template.jinja": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..YHftFVAYpY8gC2sHg6e61Q.cEqiCFtHFN_OlLG9N4SJEGa5AVGernUA9KEgnWQbCKwnIz6yhhT8mlNJKz6y7E0m5mEiHKVABSqTXJFW8sTXrCpDkUlCmh7LMQNBMrQg6ngREXbZAZX3jvCdBlKRl1n5hqVT_p92xDIhXujnVQYpJqL1HPUX0Yh6uXsuzNVnY6ztQvPVnfTtiCy7bm3ikHSFZuipV6I7VBjqE-Fbxi3anVCE5cj3j4D-PqjJGKJvk3RR1gc-gykgMDsFPbp4zabxjCRASk12RrGqxrUgtUQFI2de2Xlxe5EAO5Nlnv-SNTlBxFqGGm2jdjdh5T7qY2c4Fq0GvzUC3YRApzJgRq3j8Zzh6RC5gDm64gLliEDb384gqMEdd-B4GTMvxQICIfTliLOEbo3-ml3dGJLyo2CfMlcRLksEboGEcNg4VU4zcqscwRel2G9QT5kdo_Awckq5RZaUKFJJGaVwcn2TyWiK5H0QcjYn4xWAbJbm1VyhLFHJNly1rvMFu6gIFao3xGMpfxIWcMQ07KHRmjP8XcUXQOpAEzL5buSagGX-I1DjACWiyLtWa9CvGODtoWaHeh6lSTOLrnMVv8K34bjHKE5tilOOIlAgjHR9Dbud7f5e9LwWiwUnL42R2NrZXbzAGCEN.DwP0WJUxWu-QK3O7-gabPw/bugbounty_copilot_lora/chat_template.jinja", "bugbounty_copilot_lora/tokenizer.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..8em_6QQVzKgXIgBMOsKheg.rEJF3TZmagwMAIeVnJLxEhKUsPqYM-eQqOpCZaBHmIRSLB3D-9Bt5pb1QLgDCmXhhwxRC2ej7WURz41dLGg1q-J1QfnncRs8uj9kdWGbWA9dJi5l7egVYn5of6Y8OSs4yGbhTXp5noe3JFDer4qo96qnB_ionNRNgGNdnwWmziSlM4RWSLOYhALbwsmXArdhwCbiBYVaFXUrjoz9jMostnBK5bqhu1ASb_5aRmcxpuczRff7tlcjer3PNAu-zCjxgh3V5HFF4QV5KOIOGa7YXTixjrYSd2thJcue7gzT7oOA4uiu6PC3xPJ-hE4rkBWUUcw1xTGZxJXsyCcF1yb2rvQbsVTfvrg4mJn0TtybcbSAd8HGRbIWxCeVhM7xyglgvcaqIoPHdhG4CzLnBLdKH7P_eVg0v-csBjyOk7TwIpDAwubZ25gTr6L0-OFNCrmS7hY7PApy8ojpwQxGvBAOX7PlAsixTZ5qMQIfQQnIXEfL868H1JiryZd7ZUDQQZGHeJ03uwLyt3gy1tW0lkgeDtcU8SMgq6fwHz0qkZbXUjNnRYQuR0zE1_n8Otsza1q-HgB-b9aVNPCbY9F5_QUdHsAWwn5wJdTta_ukB6zO5cO_dk0eCxMnJtYoWpEbFZPc.crh1zw27PpE1IdBWwwco1Q/bugbounty_copilot_lora/tokenizer.json", "bugbounty_copilot_lora/tokenizer_config.json": "https://www.kaggleusercontent.com/kf/358260445/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..7AxgTzZoKAoFyi_tWTnFsg.KEE5BqbPbKz3wIg4HlRZVx1KCY9xjGrUTnCPwQsCpHYUkimN9PnYnf4fdhHW2Kzkd6f-Yc-wW993FQunmViUehsC-U9o5W0ZtOmJk5Cz4vCnZhZMi8TqgMMmdYt9uutnIPNv02rCD4ETlhgp85hDgDEZvHeMZ9eqgv8b71fVipsW4QWri_XPSE7_ei3BYzB54_KEtfv-GC88-WuAMYQRoA_wWsiFa3y97DXKyxZSlQzsCdGNQ0QfUyK_RtjgmW8xa8KPJ_cMyV_Lj35iEyuYs2bw6X93sXqzobArN-NNZDGEjXAQsMbpG1VlQnv3Bep0npr8655wWJ8qYi7w6NhyTUi4bQUUWgKkAseOwblDg_A6hBAKIIquhN6ceAdwkraz9wqEj-5SpB0djoMYkdkb4JNrM3ef6p2MpfDy9mNXN3RktyHsrepggG-kWdMv8fvVqYikX3XkLq3SF0gm-EKAXWRqS5kgyKkFVNDJZBkreyxc-K_RniVDDZUwp2SNJ-Xv9F44_32GXxj_1164fDMFUUegMWD0LuXYdwK5j73wWks2Eax0ZQFNH7BHuPFYRuQ2Chpi08LSOKiVxE6LMbEMIgl4WGKwkckE4ugU1ptwqzUM0F7IBh0UgOJfB6-_Y5bj.Oi7NXiFQKyLIiSscaEP_Cw/bugbounty_copilot_lora/tokenizer_config.json"}

possible_loras = glob.glob('/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors', recursive=True)
if not possible_loras:
    possible_loras = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

adapter_dir = None
if possible_loras:
    adapter_dir = os.path.dirname(possible_loras[0])
    print(f'[✓] Found mounted LoRA Adapter at: {adapter_dir}')
elif LATEST_LORA_URLS:
    print(f'[*] Downloading {len(LATEST_LORA_URLS)} LoRA files via signed direct URLs...')
    adapter_dir = '/kaggle/working/latest_lora'
    os.makedirs(adapter_dir, exist_ok=True)
    for fname, url in LATEST_LORA_URLS.items():
        target_file = os.path.join(adapter_dir, os.path.basename(fname))
        try:
            urllib.request.urlretrieve(url, target_file)
            print(f'    -> Downloaded: {os.path.basename(fname)}')
        except Exception as e:
            print(f'    -> Warning: {e}')

if not adapter_dir or not os.path.exists(os.path.join(adapter_dir, 'adapter_model.safetensors')):
    raise RuntimeError('Could not locate trained LoRA weights for merging!')

print(f'[✓✓✓] LoRA weights verified ready at: {adapter_dir}')


In [ ]:
# 3. Load Model in 16-bit and Fuse LoRA Weights Losslessly
from unsloth import FastLanguageModel
import torch

print('[*] Loading base model and fusing LoRA adapter into full 16-bit neural tensors...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = adapter_dir,
    max_seq_length = 4096,
    dtype = torch.float16,
    load_in_4bit = False, # Lossless 16-bit weight fusion
)
print('[✓] Model & LoRA weights successfully unified into memory graph!')


In [ ]:
# 4. Save Standalone 16-bit Model as 'Mossad-pentast-ai'
merged_dir = '/kaggle/working/Mossad-pentast-ai'
print(f'[*] Saving merged 16-bit standalone model to {merged_dir}...')
model.save_pretrained_merged(merged_dir, tokenizer, save_method = 'merged_16bit')
print(f'[✓✓✓] STANDALONE MODEL "Mossad-pentast-ai" SAVED SUCCESSFULLY!')


In [ ]:
# 5. Export Compact Standalone GGUF (Q4_K_M) for Fast Local / Server Inference
gguf_dir = '/kaggle/working/Mossad-pentast-ai-GGUF'
print(f'[*] Compiling and exporting GGUF (Q4_K_M) format to {gguf_dir}...')
try:
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method = 'q4_k_m')
    print(f'[✓✓✓] GGUF QUANTIZED MODEL "Mossad-pentast-ai" EXPORTED SUCCESSFULLY!')
except Exception as e:
    print(f'[!] GGUF compilation note ({e}). Standalone 16-bit model is 100% saved and ready.')


In [ ]:
# 6. Live Benchmark Inference Test on Merged Model 'Mossad-pentast-ai'
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "You are Mossad-pentast-ai, an Elite Offensive Security & Bug Bounty AI Copilot."},
    {"role": "user", "content": "Target: https://api.enterprise.com/v1/auth/oauth/token. Detail your systematic exploit methodology to identify host header injection, token leakage, and unauthorized password takeover."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = 'pt',
).to('cuda')

print('\n' + '='*70)
print('🎯 LIVE BENCHMARK INFERENCE TEST: Mossad-pentast-ai')
print('='*70)
outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))
print('='*70)
print('\n🎉 [FUSION COMPLETE] Mossad-pentast-ai IS FULLY BORN AND READY FOR OPERATION!')
